In [54]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/paper_block.pdf")

documents = loader.load()

print("Pages:", len(documents))

Pages: 17


In [55]:
documents

[Document(metadata={'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-04-22T10:44:29+05:30', 'moddate': '2023-04-24T17:29:45-04:00', 'ieee article id': '10101789', 'trapped': 'False', 'ieee issue id': '10005208', 'subject': 'IEEE Access;2023;11; ;10.1109/ACCESS.2023.3267047', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'ieee publication id': '6287639', 'title': 'A Survey on Consensus Algorithms in Blockchain-Based Applications: Architecture, Taxonomy, and Operational Issues', 'source': '../data/paper_block.pdf', 'total_pages': 17, 'page': 0, 'page_label': '39066'}, page_content='Received 13 March 2023, accepted 3 April 2023, date of publication 13 April 2023, date of current version 25 April 2023.\nDigital Object Identifier 10.1 109/ACCESS.2023.3267047\nA Survey on Consensus Algorithms in\nBlockchain-Based Applic

In [56]:
import re
from langchain_core.documents import Document

filtered_documents = []

for doc in documents:

    text = doc.page_content

    # Remove DOI lines
    text = re.sub(
        r"Digital Object Identifier.*?\n",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        "",
        text
    )

    # Remove received/accepted/publication dates
    text = re.sub(
        r"Received .*?current version .*?\.",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Remove copyright/license statements
    text = re.sub(
        r"This work is licensed under.*",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Remove page numbers only
    text = re.sub(
        r"\n\d{4,6}\n",
        "\n",
        text
    )

    # Remove References section (optional)
    if "REFERENCES" in text.upper():
        text = text[:text.upper().find("REFERENCES")]

    # Remove extra spaces/newlines
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    filtered_documents.append(
        Document(
            page_content=text.strip(),
            metadata={
                "title": doc.metadata.get("title"),
                "page": doc.metadata.get("page"),
                "source": doc.metadata.get("source"),
                "total_pages": doc.metadata.get("total_pages")
            }
        )
    )

print("Filtered Pages:", len(filtered_documents))

Filtered Pages: 17


In [57]:
filtered_documents

[Document(metadata={'title': 'A Survey on Consensus Algorithms in Blockchain-Based Applications: Architecture, Taxonomy, and Operational Issues', 'page': 0, 'source': '../data/paper_block.pdf', 'total_pages': 17}, page_content='A Survey on Consensus Algorithms in\nBlockchain-Based Applications: Architecture,\nTaxonomy, and Operational Issues\nSAMINUR ISLAM\n 1, MOHAMMAD JAMINUR ISLAM\n 2, MAHMUD HOSSAIN\n 3,\nSHAHID NOOR4, (Associate Member, IEEE),\nKYUNG-SUP KWAK\n 5, (Life Senior Member, IEEE),\nAND S. M. RIAZUL ISLAM\n6, (Senior Member, IEEE)\n1Department of Computer Science, North Carolina State University, Raleigh, NC 27606, USA\n2Department of Computer Science, University of California at Riverside, Riverside, CA 92521, USA\n3Department of Private Certificate Authority, Amazon Inc., Seattle, WA 98109, USA\n4Department of Computer Science, Northern Kentucky University, Highland Heights, KY 41099, USA\n5Department of Information and Communication Engineering, Inha University, Inche

In [58]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    length_function=len
)

chunks = text_splitter.split_documents(filtered_documents)

print("Total Chunks:", len(chunks))
print(chunks[0].page_content[:500])

Total Chunks: 79
A Survey on Consensus Algorithms in
Blockchain-Based Applications: Architecture,
Taxonomy, and Operational Issues
SAMINUR ISLAM
 1, MOHAMMAD JAMINUR ISLAM
 2, MAHMUD HOSSAIN
 3,
SHAHID NOOR4, (Associate Member, IEEE),
KYUNG-SUP KWAK
 5, (Life Senior Member, IEEE),
AND S. M. RIAZUL ISLAM
6, (Senior Member, IEEE)
1Department of Computer Science, North Carolina State University, Raleigh, NC 27606, USA
2Department of Computer Science, University of California at Riverside, Riverside, CA 92521, USA
3


In [59]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
# vector = embedding_model.embed_query(
#     "What are consensus algorithms?"
# )

# print(len(vector))

384


In [60]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [61]:
vector_store.save_local("vector_store")

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.load_local(
    "vector_store",
    embedding_model,
    allow_dangerous_deserialization=True
)

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = "GOOGLE_API_KEY"

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [69]:
def get_section(chunks, keywords):

    collected_text = []
    capture = False

    for chunk in chunks:

        text = chunk.page_content.upper()

        # START when keyword matches
        if not capture and any(k.upper() in text for k in keywords):
            capture = True

        # collect text once started
        if capture:
            collected_text.append(chunk.page_content)

        # STOP when a NEW section heading appears (IEEE-style detection)
        if capture:

            # detect numbered sections like I., II., III., etc.
            if any(text.strip().startswith(prefix) for prefix in
                   ["I.", "II.", "III.", "IV.", "V.", "VI.", "VII.", "VIII.", "IX."]):

                # avoid stopping on same section start
                if not any(k.upper() in text for k in keywords):
                    break

            # safety stop if end sections appear
            if any(end in text for end in [
                "REFERENCES",
                "ACKNOWLEDGMENT",
                "APPENDIX"
            ]):
                break

    return "\n\n".join(collected_text).strip()

In [70]:
import re

def route_query(query, chunks, db):

    query_lower = query.lower()

    # ---------------- ABSTRACT ----------------
    if "abstract" in query_lower:
        return get_section(chunks, ["ABSTRACT"])

    # ---------------- INTRODUCTION ----------------
    elif "introduction" in query_lower:
        docs = db.similarity_search(
            "introduction background overview blockchain cryptography",
            k=6
        )
        return "\n\n".join([d.page_content for d in docs])

    # ---------------- CONCLUSION ----------------
    elif "conclusion" in query_lower:
        docs = db.similarity_search(
            "conclusion summary future work discussion",
            k=5
        )
        return "\n\n".join([d.page_content for d in docs])

    # ---------------- FIGURE HANDLING ----------------
    elif re.search(r"figure\s*\d+", query_lower):

        fig_no = re.search(r"figure\s*(\d+)", query_lower).group(1)

        for chunk in chunks:
            if f"FIGURE {fig_no}" in chunk.page_content.upper():
                return chunk.page_content

        return "Figure not found in document."

    # ---------------- DEFAULT RAG ----------------
    else:
        docs = db.similarity_search(query, k=5)
        return "\n\n".join([d.page_content for d in docs])

In [71]:
def generate_answer(query, context):

    prompt = f"""
You are a research paper assistant.

Use ONLY the context below to answer.

Context:
{context}

Question:
{query}

Give a clear, concise answer.
"""

    response = llm.invoke(prompt)

    return response.content

In [72]:
def rag_pipeline(query, chunks, db):

    context = route_query(query, chunks, db)

    answer = generate_answer(query, context)

    return answer

In [78]:
query = "Give me the FINANCIAL MANAGEMENT ?"

result = rag_pipeline(query, chunks, db)

print(result)

In financial management, blockchain technology is a fundamental advance in financial systems and the FinTech sector, accelerated by P2P and third-party payment systems. Digital currencies utilize blockchain for faster payments, with various consensus algorithms addressing challenges like energy consumption, centralization, and economic security. Beyond cryptocurrency, blockchain transforms traditional financial systems, particularly in cross-border transactions, with the Ripple consensus algorithm popular for low-cost remittances.
